In [1]:
import sys
!{sys.executable} -m pip install folium


In [2]:
import folium
import pandas as pd
import numpy as np

# Load dataset with risk labels
df = pd.read_csv(
    r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_with_risk.csv"
)
df['datetime'] = pd.to_datetime(df['datetime'])

print("Dataset loaded!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nRisk levels: {df['predicted_risk_v2'].unique()}")

Dataset loaded!
Shape: (7446, 31)
Columns: ['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'time_gap_hours', 'winddir_sin', 'winddir_cos', 'predicted_visibility', 'city_name', 'actual_risk_v2', 'predicted_risk_v2', 'actual_recommendation', 'predicted_recommendation']

Risk levels: ['MODERATE' 'SAFE' 'LOW' 'HIGH' 'CRITICAL']


In [3]:
# Risk color mapping
risk_colors = {
    'CRITICAL': 'red',
    'HIGH': 'orange',
    'MODERATE': 'yellow',
    'LOW': 'lightgreen',
    'SAFE': 'green'
}

# City coordinates for markers
city_coords = {
    'Lahore':     [31.5204, 74.3587],
    'Islamabad':  [33.6844, 73.0479],
    'Faisalabad': [31.4504, 73.1350],
    'Multan':     [30.1575, 71.5249]
}

# Get LATEST predicted risk per city
latest_risk = df.groupby('city_name').apply(
    lambda x: x.sort_values('datetime').iloc[-1]
)[['city_name', 'datetime', 'visibility',
   'predicted_visibility', 'predicted_risk_v2',
   'predicted_recommendation']].reset_index(drop=True)

print("Latest Risk Per City:")
print(latest_risk[['city_name', 'datetime',
                    'predicted_risk_v2',
                    'predicted_recommendation']])

Latest Risk Per City:
    city_name            datetime predicted_risk_v2  \
0  Faisalabad 2025-02-28 23:00:00              SAFE   
1   Islamabad 2025-02-28 23:00:00              SAFE   
2      Lahore 2025-02-28 23:00:00          MODERATE   
3      Multan 2025-02-28 23:00:00          MODERATE   

                 predicted_recommendation  
0                       Normal Operations  
1                       Normal Operations  
2  Speed Limit 60 km/h — Caution Required  
3  Speed Limit 60 km/h — Caution Required  


C:\Users\Admin\AppData\Local\Temp\ipykernel_19468\1987777635.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  latest_risk = df.groupby('city_name').apply(


In [4]:
import json

# Path to your Flutter app's data folder
output_path = r"C:\Users\Admin\Desktop\smog-risk-prediction-system\assets\data\latest_risk.json"

# Build the list of records in the exact format the app expects
export_data = []
for _, row in latest_risk.iterrows():
    export_data.append({
        "city": row["city_name"],
        "datetime": row["datetime"].strftime("%Y-%m-%d %H:%M:%S"),
        "visibility_km": round(float(row["predicted_visibility"]), 2),
        "risk_level": row["predicted_risk_v2"],
        "recommendation": row["predicted_recommendation"]
    })

# Write to the JSON file (overwrites the existing one)
with open(output_path, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"Exported {len(export_data)} city records to:")
print(output_path)

Exported 4 city records to:
C:\Users\Admin\Desktop\smog-risk-prediction-system\assets\data\latest_risk.json


In [5]:
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background-color:white; padding:15px; border:2px solid black;
     border-radius:8px; font-family:Arial;">
<b>Smog Risk Legend</b><br>
<span style="color:red;">&#9679;</span> CRITICAL<br>
<span style="color:orange;">&#9679;</span> HIGH<br>
<span style="color:#CCCC00;">&#9679;</span> MODERATE<br>
<span style="color:lightgreen;">&#9679;</span> LOW<br>
<span style="color:green;">&#9679;</span> SAFE
</div>
"""


In [6]:
# Enhanced map with segment-wise coloring

m2 = folium.Map(
    location=[30.8, 72.5],
    zoom_start=7,
    tiles='OpenStreetMap'
)
# Get each city's current risk color dynamically
lahore_color = risk_colors[latest_risk[latest_risk['city_name']=='Lahore']['predicted_risk_v2'].values[0]]
islamabad_color = risk_colors[latest_risk[latest_risk['city_name']=='Islamabad']['predicted_risk_v2'].values[0]]
faisalabad_color = risk_colors[latest_risk[latest_risk['city_name']=='Faisalabad']['predicted_risk_v2'].values[0]]
multan_color = risk_colors[latest_risk[latest_risk['city_name']=='Multan']['predicted_risk_v2'].values[0]]

# M-2 — Lahore side
folium.PolyLine(
    locations=[[31.5204, 74.3587], [31.8000, 73.9000], [32.0800, 73.6500], [32.5000, 73.1000]],
    color=lahore_color, weight=6, opacity=0.9,
    tooltip="M-2 (Lahore side)"
).add_to(m2)

# M-2 — Islamabad side
folium.PolyLine(
    locations=[[32.5000, 73.1000], [32.9000, 72.8000], [33.3642, 73.0551]],
    color=islamabad_color, weight=6, opacity=0.9,
    tooltip="M-2 (Islamabad side)"
).add_to(m2)

# M-3 — Lahore to Faisalabad
folium.PolyLine(
    locations=[[31.5204, 74.3587], [31.2000, 73.9000], [30.9500, 73.5500], [30.6000, 73.1000], [30.5450, 72.3114]],
    color=faisalabad_color, weight=6, opacity=0.9,
    tooltip="M-3 (Lahore-Faisalabad)"
).add_to(m2)

# M-4 — Faisalabad to Multan
folium.PolyLine(
    locations=[[30.5450, 72.3114], [30.4000, 72.0000], [30.2000, 71.8000], [29.9500, 71.5500], [30.1575, 71.5249]],
    color=multan_color, weight=6, opacity=0.9,
    tooltip="M-4 (Faisalabad-Multan)"
).add_to(m2)

# City markers
for city, coords in city_coords.items():
    city_data = latest_risk[latest_risk['city_name'] == city].iloc[0]
    risk = city_data['predicted_risk_v2']
    rec = city_data['predicted_recommendation']
    vis = city_data['predicted_visibility']
    color = risk_colors[risk]

    folium.CircleMarker(
        location=coords, radius=12, color='black', fill=True,
        fill_color=color, fill_opacity=0.9, tooltip=f"{city}: {risk}",
        popup=folium.Popup(
            f"""<b>{city}</b><br>Risk: <b>{risk}</b><br>
            Predicted Visibility: {vis:.2f} km<br>Recommendation: {rec}""",
            max_width=250
        )
    ).add_to(m2)

    folium.Marker(
        location=[coords[0]+0.15, coords[1]],
        icon=folium.DivIcon(html=f'<div style="font-size:12px;font-weight:bold;">{city}</div>')
    ).add_to(m2)

# Legend
m2.get_root().html.add_child(folium.Element(legend_html))

# Save
map_path2 = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\motorway_risk_map_v2.html"
m2.save(map_path2)
print("Enhanced map saved!")

Enhanced map saved!
